# How to use the Compatibility API

The `/compatibility` endpoint displays information about the collection and returns some details about a sample granule. The output is helpful for understanding the structure of the collection and the granules so that you can craft the right set of parameters for visualization or statistics requests.

This notebook demonstrates how to use the compatibility endpoint for th [GHRSST Level 4 MUR Global Foundation Sea Surface Temperature Analysis (v4.1)](https://cmr.earthdata.nasa.gov/search/concepts/C1996881146-POCLOUD) dataset.

## Setup

In [3]:
import json
import earthaccess
import httpx

# titiler_endpoint = "https://staging.openveda.cloud/api/titiler-cmr"  # staging endpoint
titiler_endpoint = (
    "https://v4jec6i5c0.execute-api.us-west-2.amazonaws.com"  # dev endpoint
)

## Identify the dataset

You can find the MUR SST dataset using the `earthaccess.search_datasets` function.

In [13]:
datasets = earthaccess.search_datasets(concept_id="C2036881735-POCLOUD")
ds = datasets[0]

collection_concept_id = ds["meta"]["concept-id"]
print("CollectionConcept-Id: ", collection_concept_id)

print("Abstract: ", ds["umm"]["Abstract"])

CollectionConcept-Id:  C2036881735-POCLOUD
Abstract:  A Group for High Resolution Sea Surface Temperature (GHRSST) Level 4 sea surface temperature analysis, produced daily on an operational basis at the Australian Bureau of Meteorology (BoM) using optimal interpolation (OI) on a global 0.25 degree grid. This Global Australian Multi-Sensor SST Analysis (GAMSSA) v1.0 system blends satellite SST observations from passive infrared and passive microwave radiometers with in situ data from ships, drifting buoys and moorings from the Global Telecommunications System (GTS). SST observations that have experienced recent surface wind speeds less than 6 m/s during the day or less than 2 m/s during night are rejected from the analysis.  The processing results in daily foundation SST estimates that are largely free of nocturnal cooling and diurnal warming effects.  Sea ice concentrations are supplied by the NOAA/NCEP 12.7 km sea ice analysis.  In the absence of observations, the analysis relaxes to 

## Explore the collection using the `/compatibility` endpoint

In [14]:
compatibility_response = httpx.get(
    f"{titiler_endpoint}/compatibility",
    params={"collection_concept_id": collection_concept_id},
    timeout=None,
).json()

print(json.dumps(compatibility_response, indent=2))

{
  "concept_id": "C2036881735-POCLOUD",
  "backend": "xarray",
  "datetime": [
    {
      "RangeDateTimes": [
        {
          "BeginningDateTime": "2008-07-23T00:00:00.000Z"
        }
      ]
    }
  ],
  "variables": {
    "sea_ice_fraction": {
      "shape": [
        1,
        720,
        1440
      ],
      "dtype": "float64",
      "min": 0.0,
      "max": 1.0,
      "mean": 0.17563601409804552,
      "p01": 0.0,
      "p05": 0.0,
      "p95": 0.98,
      "p99": 1.0
    },
    "analysed_sst": {
      "shape": [
        1,
        720,
        1440
      ],
      "dtype": "float64",
      "min": 271.03000000000003,
      "max": 306.88000000000005,
      "mean": 287.0411285506355,
      "p01": 271.36,
      "p05": 271.36,
      "p95": 302.04,
      "p99": 302.90000000000003
    },
    "analysis_error": {
      "shape": [
        1,
        720,
        1440
      ],
      "dtype": "float64",
      "min": 0.04,
      "max": 1.0,
      "mean": 0.3044783998810321,
      "p01": 

The details from the sample granule show that it is a NetCDF file with four variables (`analysed_sst`, `analysis_error`, `mask`, and `sea_ice_fraction`) and each contains an array with a single time coordinate. The `datetime` key shows the reported temporal range from CMR which indicates that the dataset has granules from `2008-07-23` to present. For each variable several summary statistics are available to help you craft min/max values for the `rescale` parameter.

This information is handy for generating tiles via the tiles API, which you can learn more about in [the Tiles API Documentation](../tiling_api_xarray_backend_example).